# TRELLIS – Text/Image to 3D (Flash Attention 2 + احتياطي)
تشغيل [TRELLIS](https://github.com/JeffreyXiang/TRELLIS) على Google Colab.

**تحسينات:**
- ✅ تحميل الكود مباشرة من GitHub بصيغة ZIP (بدون git).
- ✅ فك الضغط باستخدام Python لضمان التوافق.
- ✅ دعم Flash Attention 2 تلقائي عند توفره.
- ✅ معالجة أخطاء شاملة في كل خطوة.

In [ ]:
import torch, subprocess, sys

print(f"PyTorch: {torch.__version__}")
if not torch.cuda.is_available():
    raise RuntimeError("الرجاء تفعيل GPU من Runtime → Change runtime type")

!nvidia-smi
gpu_name = subprocess.getoutput("nvidia-smi --query-gpu=name --format=csv,noheader")
print(f"🔹 GPU: {gpu_name}")

SUPPORTED_GPU = ["A100", "A6000", "3090", "4090", "L4", "L40", "H100"]
USE_FLASH_ATTN = any(g in gpu_name for g in SUPPORTED_GPU)
print("✨ يدعم Flash Attention 2" if USE_FLASH_ATTN else "ℹ️  يستخدم SDPA (انتباه PyTorch الافتراضي).")

In [ ]:
import pkg_resources

def is_installed(pkg):
    try:
        pkg_resources.get_distribution(pkg)
        return True
    except:
        return False

print("🔍 فحص الحزم الأساسية...")
base = ["trimesh", "pygltflib", "viser", "omegaconf", "opencv-python", "imageio", "gradio", "huggingface_hub"]
for pkg in base:
    if not is_installed(pkg):
        !pip install -q {pkg}
        print(f"✔️ {pkg}")
    else:
        print(f"✅ {pkg}")

# spconv
if not is_installed("spconv"):
    cuda_ver = torch.version.cuda.replace(".", "")
    try:
        !pip install -q spconv-cu{cuda_ver}
    except:
        !pip install -q spconv
    print("✔️ spconv")
else:
    print("✅ spconv")

# Flash Attention 2 (اختياري)
if USE_FLASH_ATTN and not is_installed("flash-attn"):
    print("⚡ تثبيت Flash Attention 2...")
    !pip install -q ninja packaging
    !pip install flash-attn --no-build-isolation
    import flash_attn
    print("✅ flash-attn")

print("\n✅ جميع الحزم الأساسية جاهزة.")

In [ ]:
import os, sys, zipfile, io, requests

# الرابط المباشر لتحميل أرشيف المستودع (لا يحتاج git)
ZIP_URL = "https://codeload.github.com/JeffreyXiang/TRELLIS/zip/refs/heads/main"
TARGET_DIR = "/content/TRELLIS-main"

if not os.path.isdir(TARGET_DIR):
    print("⬇️  تحميل المستودع من GitHub...")
    try:
        r = requests.get(ZIP_URL, timeout=30)
        r.raise_for_status()
        with zipfile.ZipFile(io.BytesIO(r.content)) as z:
            z.extractall("/content")
        print("✅ تم تنزيل وفك ضغط المستودع.")
    except Exception as e:
        print(f"❌ فشل التحميل المباشر: {e}")
        # محاولة بديلة: رابط من GitHub Releases (إن وجد)
        # سأطلب من المستخدم الرفع اليدوي
        print("يرجى تحميل الملف يدوياً من https://github.com/JeffreyXiang/TRELLIS/archive/refs/heads/main.zip ورفعه إلى Colab.")
        from google.colab import files
        uploaded = files.upload()
        for fn in uploaded.keys():
            with zipfile.ZipFile(io.BytesIO(uploaded[fn])) as z:
                z.extractall("/content")
            print("✅ تم فك الضغط.")
else:
    print("✅ المستودع موجود مسبقاً.")

# التأكد من وجود المجلد المستخرج
if os.path.isdir(TARGET_DIR):
    sys.path.insert(0, TARGET_DIR)
    print(f"📁 تم إضافة {TARGET_DIR} إلى مسار Python.")
else:
    raise FileNotFoundError("لم يتم العثور على مجلد TRELLIS-main. تأكد من نجاح التحميل.")

# استيراد trellis للتأكد
try:
    import trellis
    print("✅ استيراد trellis ناجح.")
except Exception as e:
    print(f"❌ فشل استيراد trellis: {e}")
    raise

# تحديد الانتباه
if USE_FLASH_ATTN:
    try:
        import flash_attn
        attn_impl = "flash_attention_2"
        print("🔧 Flash Attention 2 مفعل.")
    except:
        attn_impl = "sdpa"
        print("🔧 Flash Attention غير متاح، استخدام SDPA.")
else:
    attn_impl = "sdpa"
    print("🔧 SDPA مفعل.")

In [ ]:
import torch
from trellis.pipelines import TrellisImageTo3DPipeline

try:
    pipeline = TrellisImageTo3DPipeline.from_pretrained(
        "JeffreyXiang/TRELLIS-image-large",
        torch_dtype=torch.float16,
        attn_implementation=attn_impl
    )
    pipeline.to("cuda")
    print("✅ تم تحميل خط الأنابيب.")
except Exception as e:
    print(f"⚠️ خطأ أثناء التحميل: {e}")
    print("محاولة التحميل بدون تحديد attn_implementation...")
    pipeline = TrellisImageTo3DPipeline.from_pretrained(
        "JeffreyXiang/TRELLIS-image-large",
        torch_dtype=torch.float16
    )
    pipeline.to("cuda")
    print("✅ تم تحميل خط الأنابيب بالوضع الافتراضي.")

In [ ]:
from PIL import Image
import requests

url = "https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/diffusers/tiger.png"
try:
    image = Image.open(requests.get(url, stream=True).raw).convert("RGB")
    display(image)
except:
    print("⚠️ تعذر تحميل الصورة التجريبية. ارفع صورتك أو ضع رابطاً آخر.")

In [ ]:
print("⏳ جارٍ التوليد...")
try:
    outputs = pipeline.run(image, seed=42)
    gaussian = outputs.get('gaussian')
    mesh = outputs.get('mesh')
    print("🔹 تم التوليد بنجاح.")
except Exception as e:
    print(f"❌ فشل التوليد: {e}")
    gaussian, mesh = None, None

In [ ]:
from pathlib import Path
out_dir = Path("/content/outputs")
out_dir.mkdir(exist_ok=True)

if mesh:
    try:
        mesh.export(out_dir / "model.glb")
        print("✅ model.glb")
    except Exception as e:
        print(f"⚠️ فشل تصدير GLB: {e}")
else:
    print("ℹ️ لا توجد شبكة.")

if gaussian:
    try:
        if hasattr(gaussian[0], 'save_ply'):
            gaussian[0].save_ply(out_dir / "gaussian.ply")
            print("✅ gaussian.ply")
    except Exception as e:
        print(f"⚠️ فشل تصدير PLY: {e}")

In [ ]:
import viser
from google.colab.output import eval_js
from IPython.display import IFrame, display

PORT = 8080
server = viser.ViserServer(host="0.0.0.0", port=PORT)

if gaussian:
    try:
        from trellis.utils import render_utils
        render_utils.render_gaussian(server, gaussian[0])
        print("✅ تم عرض السحابة الغاوسية.")
    except ImportError:
        points = gaussian[0].xyz.detach().cpu().numpy().reshape(-1, 3)
        server.scene.add_point_cloud("/points", points=points, colors=(255,200,200), point_size=0.01)
        print("✅ تم عرض سحابة نقاط.")
else:
    print("لا توجد بيانات غاوسية.")

try:
    url = eval_js(f"google.colab.kernel.proxyPort({PORT})")
    public_url = f"https://{url}"
    print(f"🔗 رابط المشاهدة: {public_url}")
    display(IFrame(src=public_url, width="100%", height="600px"))
except Exception as e:
    print(f"⚠️ تعذر العرض المضمن: {e}")

## لماذا ستعمل هذه الطريقة؟
- استخدمنا `requests` لتحميل ملف ZIP من `codeload.github.com` (رابط ثابت ومباشر).
- فك الضغط باستخدام `zipfile.ZipFile` من Python، مما يتجنب فشل أمر `unzip`.
- في حال فشل التحميل، نوفر خيار الرفع اليدوي للملف من واجهة Colab.
- بهذه الطريقة لا نعتمد على `git` أو أوامر `unzip`.

شغّل الخلايا بالترتيب.